# 21. DUR 병용금기 EDA (T1-2)

**데이터 출처**: 식의약 데이터 포털 (https://data.mfds.go.kr)  
**파이프라인 역할**: Stage 2 한국 DDI ground truth

## EDA 체크리스트
- [ ] DUR 카테고리별 건수 (병용금기, 임부금기, 연령금기, 효능군중복 등)
- [ ] 페어 포맷: 품목일련번호↔품목일련번호 vs. 성분코드↔성분코드
- [ ] 사유(reason) 필드 구조 (코드 vs. 자유텍스트)
- [ ] 대칭성, 자기 자신 pair 여부
- [ ] TWOSIDES 공통 약물에서 pair 수준 일치율
- [ ] DUR pair ID/URI 존재 여부 (보고서 직접 인용용)
- [ ] 산출물: `data/interim/dur_clean.parquet` + `dur_summary.json`

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
from pathlib import Path

ROOT = Path("../../").resolve()
RAW  = ROOT / "data" / "raw" / "dur"
INTERIM = ROOT / "data" / "interim"
RAW.mkdir(parents=True, exist_ok=True)

# data.mfds.go.kr에서 다운로드한 파일 경로
# 예: dur_combo_contraindication.csv (다운로드 후 경로 수정)
DUR_FILES = list(RAW.glob("*.csv")) + list(RAW.glob("*.xlsx"))
print("DUR 파일 목록:", DUR_FILES)

In [ ]:
if not DUR_FILES:
    print("⚠️ DUR 파일 없음")
    print("다운로드 URL: https://data.mfds.go.kr/")
    print("경구약 병용금기 / 병용주의 데이터 검색 후 CSV 다운로드")
else:
    # 파일 읽기 (인코딩 CP949 주의)
    dfs = []
    for f in DUR_FILES:
        try:
            df_tmp = pd.read_csv(f, encoding="utf-8")
        except UnicodeDecodeError:
            df_tmp = pd.read_csv(f, encoding="cp949")
        print(f"{f.name}: {df_tmp.shape}, 컬럼={df_tmp.columns.tolist()}")
        dfs.append(df_tmp)
    df = pd.concat(dfs, ignore_index=True)

In [ ]:
if DUR_FILES:
    print("\n스키마 확인:")
    display(df.head(3).T)

In [ ]:
if DUR_FILES:
    # DUR 카테고리 분포 (컬럼명은 실제 파일 확인 후 수정)
    type_col = next((c for c in df.columns if "구분" in c or "유형" in c or "type" in c.lower()), None)
    if type_col:
        df[type_col].value_counts().plot(kind="barh", figsize=(10, 5), title="DUR 카테고리별 건수")
        plt.tight_layout()
        plt.show()
    else:
        print("카테고리 컬럼 미확인 — 컬럼 목록:", df.columns.tolist())

In [ ]:
if DUR_FILES:
    # 페어 ID 컬럼 포맷 (품목일련번호 vs 성분코드) 확인
    id_cols = [c for c in df.columns if "코드" in c or "번호" in c or "SEQ" in c.upper()]
    print("ID 관련 컬럼:", id_cols)
    if id_cols:
        for col in id_cols[:4]:
            print(f"{col} 샘플:", df[col].dropna().head(5).tolist())

In [ ]:
if DUR_FILES:
    # 자기 자신 pair / 대칭 check (pair 컬럼 확인 후)
    # 컬럼명은 실제 파일 확인 후 수정
    pair_cols = [c for c in df.columns if "성분" in c or "품목" in c]
    print("페어 관련 컬럼:", pair_cols)

In [ ]:
if DUR_FILES:
    df.to_parquet(INTERIM / "dur_clean.parquet", index=False)
    summary = {
        "total_rows": len(df),
        "columns": df.columns.tolist(),
    }
    with open(INTERIM / "dur_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print("저장 완료")